In [1]:
%pip install requests beautifulsoup4 pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 34.3 MB/s  0:00:006m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install selenium webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 43.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [selenium]/10 [selenium]ocket]er]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
from urllib.parse import urljoin, urlparse

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import fitz  # PyMuPDF


# the URL of the Haifa municipality website
BASE = "https://www.haifa.muni.il"
# the starting path for scraping - the resident services section
START_PATH = "/resident-service"

# global sets to track visited URLs and store results
visited = set()
results = []

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# function to determine if a link is internal
def is_internal_link(href):
    if href is None:
        return False
    if href.startswith("mailto:") or href.startswith("tel:"):
        return False
    if href.endswith((".jpg", ".png", ".zip", ".docx", ".xlsx")):
        return False
    # only follow links that are relative or start with the base URL
    return href.startswith("/") or href.startswith(BASE)


# function to clean and extract text from HTML, preserving hyperlinks
def clean_text(soup, base_url=None):
    # Remove unwanted tags
    for tag in soup(["script", "style", "header", "footer", "nav"]):
        tag.decompose()
    
    # Process all anchor tags to preserve hyperlinks
    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href", "").strip()
        link_text = a_tag.get_text(strip=True)
        
        # Skip empty links or special protocol links
        if not href or href.startswith("javascript:") or href.startswith("#"):
            continue
        
        # Convert relative URLs to absolute if base_url is provided
        if base_url and not href.startswith(("http://", "https://")):
            full_url = urljoin(base_url, href)
        else:
            full_url = href
        
        # Replace the anchor tag with text that includes the URL
        # Format: "link_text [URL: full_url]"
        if link_text:
            a_tag.replace_with(f"{link_text} [URL: {full_url}]")
        else:
            # If no link text, just show the URL
            a_tag.replace_with(f"[URL: {full_url}]")
    
    # Extract visible text from the page with line breaks
    text = soup.get_text(separator="\n")
    # Remove extra whitespace and empty lines
    return "\n".join([line.strip() for line in text.splitlines() if line.strip()])


# function to extract subtitle from the page
def extract_subtitle(soup):
    # case 1: specific div with class "wrap_title"
    wrap = soup.find("div", class_="wrap_title")
    if wrap:
        h1 = wrap.find("h1")
        if h1 and h1.get_text(strip=True):
            return h1.get_text(strip=True)

    # case 2: generic h1 tag
    h1 = soup.find("h1")
    if h1 and h1.get_text(strip=True):
        return h1.get_text(strip=True)

    return ""


# function to extract text from a PDF URL
def extract_pdf_text(pdf_url):
    try:
        # download the PDF file and extract text
        print("DOWNLOADING PDF:", pdf_url)
        r = requests.get(pdf_url, headers=HEADERS, timeout=15)
        # check if the request was successful
        if r.status_code != 200:
            return ""

        with open("temp.pdf", "wb") as f:
            f.write(r.content)

        doc = fitz.open("temp.pdf")
        text = ""
        # extract text from each page
        for page in doc:
            text += page.get_text()

        return text.strip()

    except Exception as e:
        print("PDF extraction failed:", pdf_url, e)
        return ""
    

# function to load a page with Selenium (for JS-rendered content)
def load_with_selenium(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
    driver.get(url)

    # wait for the page to load
    time.sleep(2)

    html = driver.page_source
    driver.quit()
    return html


# heuristic to determine if a page likely requires JS to render content
def page_requires_js(soup, text):
    # 1. if text is very short
    if len(text) < 300:
        return True
    
    # 2. if there are many empty tables
    for table in soup.find_all("table"):
        if len(table.find_all("tr")) <= 1:
            return True

    # 3. if there are many script tags with JS frameworks
    scripts = str(soup.find_all("script"))
    js_signals = ["fetch(", "axios", "v-for", ":data=", "Vue", "React"]
    if any(sig in scripts for sig in js_signals):
        return True

    return False


# main scraping function
def scrape(url):
    if url in visited:
        return
    visited.add(url)

    print("SCRAPING:", url)

    # handle PDF links
    if url.lower().endswith(".pdf"):
        pdf_text = extract_pdf_text(url)
        results.append({
            "url": url,
            "title": "PDF Document",
            "subtitle": "",
            "content": pdf_text
        })
        return
    
    # handle HTML pages via requests
    try:
        # delay to avoid overwhelming the server
        time.sleep(0.8)
        resp = requests.get(url, headers=HEADERS, timeout=10)
        html = resp.text
    except:
        return

    # parse HTML content
    soup = BeautifulSoup(html, "html.parser")

    # extract links BEFORE modifying the soup (clean_text modifies the soup)
    links_to_follow = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        links_to_follow.append(href)

    # Check if page requires JS BEFORE clean_text decomposes script tags
    # Create a temporary text extraction (without decomposing scripts) for the check
    temp_soup = BeautifulSoup(html, "html.parser")
    for tag in temp_soup(["style", "header", "footer", "nav"]):
        tag.decompose()
    temp_text = temp_soup.get_text(separator="\n")
    temp_text = "\n".join([line.strip() for line in temp_text.splitlines() if line.strip()])
    
    # selenium fallback for JS-heavy pages (check with original soup before modification)
    if page_requires_js(soup, temp_text):
        print("Using Selenium:", url)
        try:
            html = load_with_selenium(url)
            soup = BeautifulSoup(html, "html.parser")
            # extract links again after selenium load
            links_to_follow = []
            for a in soup.find_all("a", href=True):
                href = a["href"].strip()
                links_to_follow.append(href)
            text = clean_text(soup, url)
        except Exception as e:
            print("Selenium failed:", url, e)
            return
    else:
        # extract and clean text (this will modify the soup)
        text = clean_text(soup, url)

    # extract title
    title = soup.title.string.strip() if soup.title else ""
    subtitle = extract_subtitle(soup)

    # store the scraped data in json format
    results.append({
        "url": url,
        "title": title,
        "subtitle": subtitle,
        "content": text
    })

    # follow links on the page (using the links we extracted earlier)
    for href in links_to_follow:
        if not href:
            continue

        if href.lower().endswith(".pdf"):
            pdf_full = urljoin(BASE, href)
            scrape(pdf_full)
            continue

        if not is_internal_link(href):
            continue

        full = urljoin(BASE, href)
        if urlparse(full).netloc != urlparse(BASE).netloc:
            continue

        # recursively scrape the linked page
        scrape(full)


def main():
    start_url = urljoin(BASE, START_PATH)
    scrape(start_url)

    with open("haifa_scraped_with_hiperlinks.json", "w", encoding="utf8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print("DONE. Pages saved:", len(results))


if __name__ == "__main__":
    main()

SCRAPING: https://www.haifa.muni.il/resident-service
SCRAPING: https://www.haifa.muni.il/operation/security-and-emergency/
SCRAPING: https://www.haifa.muni.il/resident-service/payments/
SCRAPING: https://www.haifa.muni.il/resident-service/service-center/
SCRAPING: https://www.haifa.muni.il/resident-service/arnona/
SCRAPING: https://www.haifa.muni.il/resident-service/miluim/
SCRAPING: https://www.haifa.muni.il/operation/boars/
SCRAPING: https://www.haifa.muni.il/education/registration-center/
SCRAPING: https://www.haifa.muni.il/resident-service/pharmacy/
SCRAPING: https://www.haifa.muni.il/operation/security-and-emergency/safe-place/
SCRAPING: https://www.haifa.muni.il/operation/security-and-emergency/emergency-preparedness/missile-attack/
SCRAPING: https://www.haifa.muni.il/operation/security-and-emergency/emergency-preparedness/hazardous-materials/
SCRAPING: https://www.haifa.muni.il/operation/security-and-emergency/emergency-preparedness/
SCRAPING: https://www.haifa.muni.il/article/f

/tmp/ipykernel_173473/687128251.py:182: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "html.parser")
/tmp/ipykernel_173473/687128251.py:192: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing m

SCRAPING: https://www.haifa.muni.il/event/haifahag/
SCRAPING: https://www.haifa.muni.il/event/%d7%a1%d7%95%d7%9c%d7%a1%d7%98%d7%90%d7%a8%d7%a1-%d7%97%d7%a0%d7%95%d7%9b%d7%94-2025/
SCRAPING: https://www.haifa.muni.il/event/%d7%97%d7%92%d7%99%d7%92%d7%aa-%d7%97%d7%a0%d7%95%d7%9b%d7%94-%d7%9c%d7%9e%d7%a9%d7%a4%d7%97%d7%95%d7%aa-%d7%a6%d7%a2%d7%99%d7%a8%d7%95%d7%aa/
SCRAPING: https://www.haifa.muni.il/event_cat/%d7%91%d7%99%d7%aa-%d7%90%d7%91%d7%90-%d7%97%d7%95%d7%a9%d7%99/
SCRAPING: https://www.haifa.muni.il/event_cat/cinema/
SCRAPING: https://www.haifa.muni.il/event_cat/roundel/
SCRAPING: https://www.haifa.muni.il/event_cat/%d7%91%d7%99%d7%aa-%d7%a0%d7%92%d7%9c%d7%a8/
SCRAPING: https://www.haifa.muni.il/event_cat/%d7%94%d7%a8%d7%a6%d7%90%d7%95%d7%aa-%d7%91%d7%91%d7%99%d7%aa-%d7%a0%d7%92%d7%9c%d7%a8/
SCRAPING: https://www.haifa.muni.il/event_cat/%d7%a4%d7%a2%d7%99%d7%9c%d7%95%d7%99%d7%95%d7%aa-%d7%94%d7%a4%d7%92%d7%94/
SCRAPING: https://www.haifa.muni.il/event_cat/%d7%98%d7%91%d7%a2-%d7%9